In [19]:
import re
import numpy as np

np.random.seed(42)

In [24]:
corpus = """
The quick brown fox jumps over the lazy dog.
The dog barks at the fox and the fox runs away.
A quick brown dog jumps over the lazy fox.
Natural language processing helps computers understand human language.
Word embeddings capture the meaning of words in vectors.
The model learns word vectors by predicting a word from its context.
Computers learn language from large amounts of text.
The cat and the dog are friends and the cat likes the dog.
"""

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)     
    return text.split()                         

tokens = preprocess(corpus)
vocab = sorted(set(tokens))
V = len(vocab)
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print("=== a. Data preparation ===")
print(f"Total tokens : {len(tokens)}")
print(f"Vocab size   : {V}")
print(f"Vocabulary   : {vocab}\n")

=== a. Data preparation ===
Total tokens : 79
Vocab size   : 44
Vocabulary   : ['a', 'amounts', 'and', 'are', 'at', 'away', 'barks', 'brown', 'by', 'capture', 'cat', 'computers', 'context', 'dog', 'embeddings', 'fox', 'friends', 'from', 'helps', 'human', 'in', 'its', 'jumps', 'language', 'large', 'lazy', 'learn', 'learns', 'likes', 'meaning', 'model', 'natural', 'of', 'over', 'predicting', 'processing', 'quick', 'runs', 'text', 'the', 'understand', 'vectors', 'word', 'words']



In [25]:
WINDOW = 2   

def generate_training_data(tokens, window):
    data = []
    for i, target in enumerate(tokens):
        context = [tokens[j]
                   for j in range(max(0, i - window), min(len(tokens), i + window + 1))
                   if j != i]
        data.append((context, target))
    return data

def one_hot(idx, size):
    v = np.zeros(size)
    v[idx] = 1
    return v

pairs = generate_training_data(tokens, WINDOW)
# X: (context indices list), y: target index
X = [[word2idx[w] for w in ctx] for ctx, _ in pairs]
y = [word2idx[t] for _, t in pairs]

print("=== b. Training data (first 5 samples) ===")
for ctx, tgt in pairs[:5]:
    print(f"  context={ctx}  ->  target='{tgt}'")
print(f"Total training samples: {len(pairs)}\n")

=== b. Training data (first 5 samples) ===
  context=['quick', 'brown']  ->  target='the'
  context=['the', 'brown', 'fox']  ->  target='quick'
  context=['the', 'quick', 'fox', 'jumps']  ->  target='brown'
  context=['quick', 'brown', 'jumps', 'over']  ->  target='fox'
  context=['brown', 'fox', 'over', 'the']  ->  target='jumps'
Total training samples: 79



In [27]:
N = 10    
EPOCHS = 500
LR = 0.05

W1 = np.random.randn(V, N) * 0.1   
W2 = np.random.randn(N, V) * 0.1  

def softmax(u):
    e = np.exp(u - np.max(u))
    return e / e.sum()

def forward(context_idx):
    x_avg = np.mean([one_hot(i, V) for i in context_idx], axis=0)
    h = x_avg @ W1                  
    u = h @ W2                      
    y_hat = softmax(u)           
    return x_avg, h, y_hat

print("=== c. Training ===")
loss_history = []
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    for ctx_idx, tgt_idx in zip(X, y):
        x_avg, h, y_hat = forward(ctx_idx)
        total_loss += -np.log(y_hat[tgt_idx] + 1e-9)

        e = y_hat.copy()
        e[tgt_idx] -= 1                      
        dW2 = np.outer(h, e)                 
        dh = W2 @ e                           
        dW1 = np.outer(x_avg, dh)             
        W2 -= LR * dW2
        W1 -= LR * dW1

    loss_history.append(total_loss / len(X))
    if epoch % 20 == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS}  loss = {loss_history[-1]:.4f}")

=== c. Training ===
  Epoch    1/500  loss = 3.7839
  Epoch   20/500  loss = 3.3775
  Epoch   40/500  loss = 2.5463
  Epoch   60/500  loss = 1.8645
  Epoch   80/500  loss = 1.1705
  Epoch  100/500  loss = 0.7079
  Epoch  120/500  loss = 0.4586
  Epoch  140/500  loss = 0.3253
  Epoch  160/500  loss = 0.2483
  Epoch  180/500  loss = 0.2000
  Epoch  200/500  loss = 0.1673
  Epoch  220/500  loss = 0.1437
  Epoch  240/500  loss = 0.1258
  Epoch  260/500  loss = 0.1119
  Epoch  280/500  loss = 0.1012
  Epoch  300/500  loss = 0.0929
  Epoch  320/500  loss = 0.0865
  Epoch  340/500  loss = 0.0815
  Epoch  360/500  loss = 0.0776
  Epoch  380/500  loss = 0.0744
  Epoch  400/500  loss = 0.0718
  Epoch  420/500  loss = 0.0696
  Epoch  440/500  loss = 0.0678
  Epoch  460/500  loss = 0.0662
  Epoch  480/500  loss = 0.0648
  Epoch  500/500  loss = 0.0635


In [29]:
print("\n=== d. Output ===")

def predict(context_words, top_k=3):
    idxs = [word2idx[w] for w in context_words]
    _, _, y_hat = forward(idxs)
    top = np.argsort(y_hat)[::-1][:top_k]
    return [(idx2word[i], float(y_hat[i])) for i in top]

def most_similar(word, top_k=3):
    emb = W1 / (np.linalg.norm(W1, axis=1, keepdims=True) + 1e-9)
    sims = emb @ emb[word2idx[word]]
    best = np.argsort(sims)[::-1][1:top_k + 1]
    return [(idx2word[i], float(sims[i])) for i in best]

tests = [["the", "quick", "fox", "jumps"],
         ["the", "lazy", "the", "barks"],
         ["word", "capture", "meaning", "of"]]
for ctx in tests:
    print(f"Context {ctx} -> predicted: {predict(ctx)}")

print("\nEmbedding of 'fox':")
print(np.round(W1[word2idx['fox']], 3))

print("\nMost similar words:")
for w in ["fox", "language", "dog"]:
    print(f"  {w}: {[(a, round(b, 3)) for a, b in most_similar(w)]}")

print(f"\nFinal loss: {loss_history[-1]:.4f} (started at {loss_history[0]:.4f})")


=== d. Output ===
Context ['the', 'quick', 'fox', 'jumps'] -> predicted: [('brown', 0.9875785035448895), ('over', 0.0053186892020125965), ('dog', 0.002131482866545367)]
Context ['the', 'lazy', 'the', 'barks'] -> predicted: [('dog', 0.997452921797369), ('fox', 0.0023343726766447185), ('cat', 0.00012222019249988946)]
Context ['word', 'capture', 'meaning', 'of'] -> predicted: [('words', 0.7931958507693931), ('amounts', 0.06711663856686402), ('in', 0.061217649169933615)]

Embedding of 'fox':
[ 4.775  1.835  1.554 -0.819 -0.671 -2.418 -1.665 -0.251 -4.809  0.783]

Most similar words:
  fox: [('embeddings', 0.761), ('cat', 0.654), ('learn', 0.648)]
  language: [('by', 0.557), ('processing', 0.527), ('capture', 0.465)]
  dog: [('cat', 0.63), ('fox', 0.58), ('brown', 0.437)]

Final loss: 0.0635 (started at 3.7839)
